# Gradient Boosting in scikit-learn

## Why this notebook exists

This lesson jumps straight from AdaBoost to XGBoost. XGBoost is the right tool for a Kaggle
leaderboard, but the jump skips two things worth having:

1. **The mechanism.** Gradient boosting fits each new tree to the *residuals* of the ensemble
   so far. That one sentence explains the whole family, and it is easier to see in
   scikit-learn's plain implementation than through XGBoost's regularised, distributed one.
2. **The estimator you will actually reach for.** `HistGradientBoostingClassifier` is in
   scikit-learn, needs no extra dependency, handles missing values natively, and is fast
   enough that on most datasets XGBoost buys you very little.

XGBoost stays the lesson's headline algorithm. This notebook supplies what is underneath it.

In [ ]:
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from sklearn.compose import make_column_selector, make_column_transformer
from sklearn.ensemble import (
    GradientBoostingRegressor,
    HistGradientBoostingClassifier,
    RandomForestClassifier,
    RandomForestRegressor,
)
from sklearn.metrics import accuracy_score, mean_squared_error, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder
from sklearn.tree import DecisionTreeRegressor

RSEED = 42

## Part 1 — boosting is fitting the leftovers

Build it by hand on a tiny problem first. One feature, an obviously curved relationship, and
trees so shallow they can barely do anything (`max_depth=1` — a single split, often called a
**stump**).

In [ ]:
rng = np.random.default_rng(RSEED)
x = np.linspace(0, 10, 120)
y_true = np.sin(x) * 3 + x * 0.5
y_obs = y_true + rng.normal(0, 0.4, size=x.shape)
X_toy = x.reshape(-1, 1)

plt.scatter(x, y_obs, s=15, alpha=0.6, color="#7a7a7a", label="observed")
plt.plot(x, y_true, color="#1A1A1A", lw=2, label="true relationship")
plt.legend()
plt.title("A curve one stump has no hope of fitting");

### The algorithm, in eight lines

Start by predicting the mean. Then repeatedly: look at what is still wrong (the residuals),
fit a stump to *those*, and add a fraction of its prediction to the running total.

That fraction is the **learning rate**.

In [ ]:
learning_rate = 0.3
n_rounds = 60

prediction = np.full_like(y_obs, y_obs.mean())    # round 0: just the mean
stumps, snapshots = [], {}

for round_number in range(1, n_rounds + 1):
    residuals = y_obs - prediction                # what we still get wrong
    stump = DecisionTreeRegressor(max_depth=1, random_state=RSEED)
    stump.fit(X_toy, residuals)                   # fit the leftovers
    prediction = prediction + learning_rate * stump.predict(X_toy)
    stumps.append(stump)
    if round_number in (1, 5, 20, 60):
        snapshots[round_number] = prediction.copy()

print(f"{len(stumps)} stumps, each fitted to the previous ensemble's mistakes")

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 3.6), sharey=True)

for ax, (round_number, pred) in zip(axes, snapshots.items()):
    ax.scatter(x, y_obs, s=10, alpha=0.4, color="#7a7a7a")
    ax.plot(x, y_true, color="#1A1A1A", lw=1.5, ls=":")
    ax.plot(x, pred, color="#F5821F", lw=2.5)
    rmse = np.sqrt(mean_squared_error(y_obs, pred))
    ax.set_title(f"after {round_number} round{'s' if round_number > 1 else ''}\nRMSE {rmse:.3f}", fontsize=10)

plt.tight_layout()
plt.show()

A staircase assembling itself out of single splits. No individual stump is worth anything —
each one is a horizontal line with one step in it — but sixty of them, each correcting the
last, trace the curve.

**That is the whole idea.** Compare it against bagging from the previous lesson:

| | **Bagging** (random forest) | **Boosting** |
|---|---|---|
| trees are built | independently, in parallel | in sequence, each fixing the last |
| each tree sees | a bootstrap sample of the data | the current residuals |
| combined by | averaging | adding, scaled by the learning rate |
| base learners are | deep, low-bias, high-variance | shallow, high-bias, low-variance |
| reduces | **variance** | **bias** |
| more trees | never hurts | **eventually overfits** |

That last row is the practical one, and it is why boosting needs tuning where bagging did
not.

## Part 2 — `GradientBoostingRegressor`

scikit-learn's implementation of exactly what we just wrote.

In [ ]:
gbr = GradientBoostingRegressor(
    n_estimators=60, learning_rate=0.3, max_depth=1, random_state=RSEED
)
gbr.fit(X_toy, y_obs)

print(f"our hand-rolled version : RMSE {np.sqrt(mean_squared_error(y_obs, prediction)):.4f}")
print(f"GradientBoostingRegressor: RMSE {np.sqrt(mean_squared_error(y_obs, gbr.predict(X_toy))):.4f}")

Identical to four decimal places. The eight lines above really are the algorithm — 
scikit-learn's version adds subsampling, several loss functions and a lot of speed, but the
mechanism is what you just wrote.

### `learning_rate` and `n_estimators` trade off

This is the pairing that makes boosting harder to tune than bagging. A smaller learning rate
takes smaller steps, so it needs **more** rounds to get anywhere — but it usually generalises
better, because each step commits less.

In [ ]:
X_tr, X_te, y_tr, y_te = train_test_split(X_toy, y_obs, test_size=0.25, random_state=RSEED)

for lr in [0.01, 0.1, 0.5, 1.0]:
    row = []
    for n in [10, 50, 200, 500]:
        m = GradientBoostingRegressor(n_estimators=n, learning_rate=lr,
                                      max_depth=1, random_state=RSEED).fit(X_tr, y_tr)
        row.append(f"{np.sqrt(mean_squared_error(y_te, m.predict(X_te))):.3f}")
    print(f"learning_rate={lr:<5} test RMSE at n=10,50,200,500: {'  '.join(row)}")

Read along each row. A tiny learning rate is still improving at 500 rounds; a large one
reaches its best early and then **gets worse** — the overfitting that bagging could not do.

The two parameters are not independent, which is why they must be searched **together** —
exactly the situation the Hyperparameter Tuning lesson's grid and randomized searches are
for.

## Part 3 — `HistGradientBoosting`, the one to actually use

Classic gradient boosting considers every possible split point on every feature, which is
slow on large data. The **histogram** variant first bins each feature into at most 255
buckets, then only considers bin edges as candidate splits.

The approximation costs almost nothing in accuracy and buys a large speed-up. It also
**handles missing values natively** — no imputer required.

Try it on the census data from this lesson's other notebooks: 45,222 adults, predict whether
income exceeds \$50K.

In [ ]:
census = pd.read_csv("data/census.csv")

X = census.drop(columns="income")
y = (census["income"].str.strip() == ">50K").astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=RSEED, stratify=y
)

print(f"{len(X_train):,} training rows, {X.shape[1]} features")
print(f"earning >50K: {y.mean():.1%}")

In [ ]:
# Trees split on ordered values, so ordinal encoding is fine here — and it keeps the
# feature count down. A linear model would need one-hot; see the Linear Regression lesson.
categorical = make_column_selector(dtype_exclude="number")

preprocessor = make_column_transformer(
    (OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1), categorical),
    remainder="passthrough",
)

models = {
    "random forest": RandomForestClassifier(n_estimators=100, random_state=RSEED, n_jobs=-1),
    "hist gradient boosting": HistGradientBoostingClassifier(random_state=RSEED),
}

for name, clf in models.items():
    pipe = make_pipeline(preprocessor, clf)
    start = time.perf_counter()
    pipe.fit(X_train, y_train)
    elapsed = time.perf_counter() - start
    auc = roc_auc_score(y_test, pipe.predict_proba(X_test)[:, 1])
    print(f"{name:24s} fit {elapsed:5.1f}s   test ROC-AUC {auc:.4f}   "
          f"accuracy {accuracy_score(y_test, pipe.predict(X_test)):.4f}")

### Early stopping

Because a boosted model *can* overfit with more rounds, it is worth letting it decide when to
stop. `early_stopping=True` holds out a slice of the training data and halts once the score
on it stops improving.

This is how you avoid tuning `n_estimators` by hand.

In [ ]:
hgb = HistGradientBoostingClassifier(
    max_iter=1000,
    early_stopping=True,
    n_iter_no_change=15,
    validation_fraction=0.1,
    random_state=RSEED,
)
pipe = make_pipeline(preprocessor, hgb)
pipe.fit(X_train, y_train)

print(f"max_iter allowed : 1000")
print(f"trees actually built: {hgb.n_iter_}")
print(f"test ROC-AUC     : {roc_auc_score(y_test, pipe.predict_proba(X_test)[:, 1]):.4f}")

It stopped well short of the 1,000 it was allowed, having decided further trees were not
helping. **Set `max_iter` generously and let early stopping find the number** — it is more
reliable than guessing, and cheaper than searching over it.

## Where this leaves XGBoost

XGBoost implements the same algorithm with extra regularisation terms, a different split
criterion and its own handling of sparsity. On a large, messy, competitive problem those
details win competitions.

For everyday work `HistGradientBoostingClassifier` is usually within a whisker of it, ships
with scikit-learn, and needs no separate install. **Reach for it first**; reach for XGBoost
when you have a reason.

## Summary

- Boosting fits each new learner to the **residuals** of the ensemble so far, adding a
  `learning_rate` fraction of it each round.
- Bagging reduces **variance** with deep independent trees; boosting reduces **bias** with
  shallow sequential ones.
- `learning_rate` and `n_estimators` trade off and must be tuned **together**.
- Unlike a forest, a boosted model **will** overfit with too many rounds.
- `HistGradientBoosting*` bins features for speed, handles missing values natively, and
  supports **early stopping** so you need not tune the number of rounds at all.

> **Where this goes next.** The **Advanced ML-Pipelines** lesson puts models like this inside
> a search over preprocessing and model choice together. The `learning_rate` × `n_estimators`
> interaction is the standard worked example for why a grid must be searched jointly.